In [ ]:
# === CELL 1: SETUP & CONFIGURATION ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm  # For MobileNetV3 Backbone
import numpy as np
import os
import cv2
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast # Crucial for 3080 Ti speed

# Hardware Check
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE} (AMP Enabled)")

# GLOBAL CONFIGURATION
CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,
    'input_channels': 1, # Grayscale
    
    # Stereo Geometry
    'max_disp_pixel': 192,         # Max disparity in original image pixels
    'backbone_stride': 4,          # MobileNet Stage 1 output
    'internal_disp_steps': 48,     # 192 / 4 = 48 steps for cost volume
    
    # Heads
    'num_seg_classes': 6,          # 5 Robotics + Background
    'num_det_classes': 80,         # COCO standard
    
    # Training
    'batch_size': 8,               # Adjust for 3080 Ti (12GB VRAM)
    'lr_backbone': 1e-4,
    'lr_heads': 3e-4,
}

print(f"ℹ️  Configuration Loaded. Max Disparity: {CONFIG['max_disp_pixel']}px ({CONFIG['internal_disp_steps']} internal steps)")

In [ ]:
# === CELL 2: DATA LOADING PIPELINE ===
class FusedHexapodDataset(Dataset):
    def __init__(self, root_dir, mode='train', task='stereo', transform=None):
        """
        mode: 'train' or 'val'
        task: 'stereo' (FT3D), 'robotics' (TartanAir), 'coco' (Detection)
        """
        self.root = root_dir
        self.mode = mode
        self.task = task
        self.transform = transform
        self.file_list = self._scan_files()

    def _scan_files(self):
        # Placeholder: Implement actual file scanning logic here
        # Return list of dicts: [{'left': path, 'right': path, 'disp': path, 'seg': path, ...}]
        return [] 

    def load_disp(self, path):
        # Handle .pfm or .png disparity loading
        if path.endswith('.pfm'):
            # Custom PFM loader would go here
            return np.zeros((480, 640), dtype=np.float32)
        return cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0

    def __len__(self):
        return 100 # Dummy length for testing

    def __getitem__(self, idx):
        # 1. Load Grayscale Images
        # In real code: Load from self.file_list[idx]
        # Simulating data for architecture verification:
        left = torch.randn(1, 480, 640) 
        right = torch.randn(1, 480, 640)
        
        targets = {}
        
        if self.task == 'stereo':
            # GT Disparity
            targets['disp'] = torch.rand(480, 640) * 192.0
            targets['seg'] = None
            targets['det'] = None
            
        elif self.task == 'robotics':
            # TartanAir Seg + Det
            targets['disp'] = None # or sparse
            targets['seg'] = torch.randint(0, 6, (480, 640)).long()
            targets['det'] = torch.zeros((10, 5)) # [x,y,w,h,cls]
            
        return {'left': left, 'right': right, 'targets': targets}

# Test the shape
ds = FusedHexapodDataset("dummy", task='stereo')
sample = ds[0]
print(f"📦 Data Shape Check -> Left: {sample['left'].shape}")

In [ ]:
# === CELL 3: MODEL ARCHITECTURE (NPU OPTIMIZED) ===
# === CELL 3: COARSE-TO-FINE MODEL ARCHITECTURE ===
# === CELL 3: COARSE-TO-FINE MODEL ARCHITECTURE (UPDATED) ===
# === CELL 3: COARSE-TO-FINE MODEL (FINAL OPTIMIZED) ===

class ConvMean(nn.Module):
    """
    Replaces torch.mean(dim=1) with a fixed 1x1 Conv.
    This ensures the reduction is strictly 'Conv-Expressible' for the NPU.
    """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, 1, bias=False)
        # Fixed weights = 1/C
        with torch.no_grad():
            self.conv.weight.fill_(1.0 / in_channels)
        self.conv.weight.requires_grad = False

    def forward(self, x):
        return self.conv(x)

class SimpleResidualCorrelation(nn.Module):
    """
    Final Recommendation: Small D + Loop + Immediate Reduction.
    Avoids massive memory expansion of vectorized approaches.
    """
    def __init__(self, in_channels, max_disp):
        super().__init__()
        self.D = max_disp
        self.inter_channels = 16
        
        # 1. Reduce Channels first (Save Bandwidth)
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_channels, 1, bias=False),
            nn.BatchNorm2d(self.inter_channels),
            nn.ReLU(inplace=True)
        )
        
        # 2. NPU-Friendly Reducer (ConvMean instead of generic mean)
        self.channel_reducer = ConvMean(in_channels=self.inter_channels) 

    def forward(self, left, right):
        # Reduce input feature depth
        l = self.reduce(left) # [B, 16, H, W]
        r = self.reduce(right)
        B, C, H, W = l.shape
        
        cost_stack = []
        for d in range(self.D):
            if d > 0:
                # 1. Shift Right Image (Slice)
                # Matches Right(x-d) against Left(x)
                # Slice off the last 'd' columns
                r_shifted = r[:, :, :, :-d]
                
                # 2. Multiply (Correlation)
                # Corresponding Left region
                l_region = l[:, :, :, d:]
                product = l_region * r_shifted
                
                # 3. Immediate Reduction (16ch -> 1ch)
                # This is the key: We never materialize [B, 16, H, W] for all D.
                sim = self.channel_reducer(product)
                
                # 4. Pad back to original width (Left Padding)
                # (d, 0, 0, 0) -> Pad left by d, right by 0
                sim_padded = F.pad(sim, (d, 0, 0, 0), mode='constant', value=0)
                cost_stack.append(sim_padded)
            else:
                # Zero shift
                product = l * r
                sim = self.channel_reducer(product)
                cost_stack.append(sim)

        # Resulting cost volume: [B, D, H, W]
        return torch.cat(cost_stack, dim=1)

class StereoHeadCoarseFine(nn.Module):
    def __init__(self, ch_s16, ch_s4):
        super().__init__()
        
        # --- STAGE 1: COARSE (Stride 16) ---
        # Search Range: 192/16 = 12 pixels
        self.D_coarse = 12 
        self.corr_coarse = SimpleResidualCorrelation(ch_s16, self.D_coarse)
        
        self.refine_coarse = nn.Sequential(
            nn.Conv2d(self.D_coarse, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, self.D_coarse, 3, 1, 1, bias=False) 
        )
        
        # Fixed Weights for Soft-Argmin (Coarse) -> Conv Implementation
        self.coarse_soft_sum = nn.Conv2d(self.D_coarse, 1, 1, bias=False)
        with torch.no_grad():
            self.coarse_soft_sum.weight.data = torch.arange(self.D_coarse).float().view(1, self.D_coarse, 1, 1)
        self.coarse_soft_sum.weight.requires_grad = False

        # --- STAGE 2: FINE (Stride 4) ---
        # Search Range: +/- 8 pixels (Total 17) -> Safety Buffer
        self.D_fine = 17 
        self.corr_fine = SimpleResidualCorrelation(ch_s4, self.D_fine)
        
        self.refine_fine = nn.Sequential(
            nn.Conv2d(self.D_fine + 1, 32, 3, 1, 1, bias=False), # +1 for coarse hint
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, self.D_fine, 3, 1, 1, bias=False) 
        )

    def warp(self, x, disp):
        """ Warp 'x' (Right) towards Left based on 'disp'. """
        B, _, H, W = x.size()
        # Create Grid
        xx = torch.arange(0, W, device=x.device).view(1, -1).repeat(H, 1)
        yy = torch.arange(0, H, device=x.device).view(-1, 1).repeat(1, W)
        xx = xx.view(1, 1, H, W).repeat(B, 1, 1, 1).float()
        yy = yy.view(1, 1, H, W).repeat(B, 1, 1, 1).float()
        
        # Apply Disparity (subtract from X)
        grid_x = xx - disp.squeeze(1)
        
        # Normalize to [-1, 1]
        grid_x = 2.0 * grid_x / max(W - 1, 1) - 1.0
        grid_y = 2.0 * yy / max(H - 1, 1) - 1.0
        
        grid = torch.stack((grid_x, grid_y), dim=3) # [B, H, W, 2]
        
        return F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=True)

    def forward(self, feat_l_s16, feat_r_s16, feat_l_s4, feat_r_s4):
        # 1. Coarse Pass
        cost_coarse = self.corr_coarse(feat_l_s16, feat_r_s16)
        cost_coarse = self.refine_coarse(cost_coarse) # [B, 12, H16, W16]
        
        # Coarse Disparity (Soft Argmin via Conv)
        prob_coarse = F.softmax(cost_coarse, dim=1)
        disp_coarse_norm = self.coarse_soft_sum(prob_coarse) # [B, 1, H16, W16]
        
        # Upsample Coarse to S4 resolution
        disp_coarse_s4 = F.interpolate(disp_coarse_norm, size=feat_l_s4.shape[-2:], mode='bilinear', align_corners=False)
        disp_coarse_s4 = disp_coarse_s4 * 4.0 
        
        # 2. Fine Pass
        feat_r_warped = self.warp(feat_r_s4, disp_coarse_s4)
        cost_fine = self.corr_fine(feat_l_s4, feat_r_warped) # [B, 17, H4, W4]
        
        # Concatenate coarse hint
        cost_fine = torch.cat([cost_fine, disp_coarse_s4], dim=1) # [B, 18, H4, W4]
        cost_fine = self.refine_fine(cost_fine) # [B, 17, H4, W4]
        
        return cost_coarse, cost_fine, disp_coarse_s4

class FusedHexapodModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 3, 4))
        self.stereo_head = StereoHeadCoarseFine(ch_s16=112, ch_s4=24)
        self.seg_head = nn.Conv2d(24, 6, 1) # Placeholder
        self.yolo_head = nn.ModuleList([nn.Conv2d(40, 85, 1)]) # Placeholder

    def forward(self, left, right=None):
        x = left.repeat(1, 3, 1, 1)
        feats_l = self.backbone(x) # [s4, s8, s16, s32]
        
        cost_coarse, cost_fine, disp_coarse = None, None, None
        
        if right is not None:
            xr = right.repeat(1, 3, 1, 1)
            feats_r = self.backbone(xr)
            cost_coarse, cost_fine, disp_coarse = self.stereo_head(feats_l[2], feats_r[2], feats_l[0], feats_r[0])
            
        return cost_coarse, cost_fine, disp_coarse

model = FusedHexapodModel().to(DEVICE)
print(f"✅ Final Optimized Model Built. (Simple Residual Correlation + ConvMean)")
print(f"✅ Coarse-to-Fine Model Built. Fine Search Range: ±8 pixels (Total 17).")
print("✅ Coarse-to-Fine NPU Model Built.")
print("✅ NPU-Compatible Model Built (Parallel Cost Volume).")
print("✅ Ready for Training.")

In [ ]:
# === CELL 4: LOSS FUNCTIONS & POST-PROCESSING ===

class StereoPostProcessor:
    @staticmethod
    def process(cost_fine, disp_coarse_s4, min_disp=-8):
        # 1. Softmax
        prob = F.softmax(cost_fine, dim=1) 
        
        # 2. Residual Regression (-8 to +8)
        shifts = torch.arange(cost_fine.shape[1], device=cost_fine.device).float() + min_disp
        shifts = shifts.view(1, -1, 1, 1)
        disp_residual = torch.sum(prob * shifts, dim=1, keepdim=True)
        
        # 3. Combine & Scale
        final_disp_s4 = disp_coarse_s4 + disp_residual
        final_disp = F.interpolate(final_disp_s4, scale_factor=4, mode='bilinear', align_corners=False)
        return final_disp * 4.0 

class HexapodLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.ce = nn.CrossEntropyLoss(ignore_index=255)
        self.processor = StereoPostProcessor()
        
    def sobel_gradient(self, img):
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], device=img.device).float().view(1,1,3,3)
        ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], device=img.device).float().view(1,1,3,3)
        gx = F.conv2d(img, kx, padding=1)
        gy = F.conv2d(img, ky, padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-6)

    def forward(self, preds, targets):
        stereo_out, seg_pred, det_pred = preds
        cost_coarse, cost_fine, disp_coarse_s4 = stereo_out
        t_disp, t_seg = targets.get('disp'), targets.get('seg')
        
        total_loss = 0
        loss_dict = {}
        
        if t_disp is not None and cost_fine is not None:
            # Reconstruct Disparity
            disp_pred = self.processor.process(cost_fine, disp_coarse_s4, min_disp=-8)
            
            mask = (t_disp > 0) & (t_disp < CONFIG['max_disp_pixel'])
            if mask.sum() > 0:
                l1 = F.smooth_l1_loss(disp_pred[mask], t_disp[mask])
                
                edge_pred = self.sobel_gradient(disp_pred)
                edge_gt = self.sobel_gradient(t_disp)
                l_edge = F.l1_loss(edge_pred[mask], edge_gt[mask])
                
                loss_main = l1 + 0.5 * l_edge
                
                # Aux Loss (Coarse)
                prob_c = F.softmax(cost_coarse, dim=1)
                idx_c = torch.arange(cost_coarse.shape[1], device=cost_coarse.device).float().view(1,-1,1,1)
                disp_c_full = F.interpolate(torch.sum(prob_c * idx_c, dim=1, keepdim=True), size=t_disp.shape[-2:], mode='bilinear') * 16.0
                l_coarse = F.smooth_l1_loss(disp_c_full[mask], t_disp[mask])
                
                loss_stereo = loss_main + 0.3 * l_coarse
                total_loss += loss_stereo
                loss_dict['stereo'] = loss_stereo.item()

        if t_seg is not None and seg_pred is not None:
            l_seg = self.ce(seg_pred, t_seg.long())
            total_loss += l_seg
            loss_dict['seg'] = l_seg.item()
            
        return total_loss, loss_dict

criterion = HexapodLoss().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr_backbone'])
scaler = GradScaler()
print("✅ Loss Function Loaded.")

In [ ]:
# === CELL 5: TRAINING LOOP ===
def train_epoch(model, loader, stage='stereo_only'):
    model.train()
    
    # FREEZE LOGIC
    if stage == 'stereo_only':
        for p in model.backbone.parameters(): p.requires_grad = True
        for p in model.stereo.parameters(): p.requires_grad = True
        for p in model.seg.parameters(): p.requires_grad = False
        for p in model.yolo_layers.parameters(): p.requires_grad = False
    elif stage == 'distill_heads':
        # Freeze backbone and stereo to preserve geometry
        for p in model.backbone.parameters(): p.requires_grad = False
        for p in model.stereo.parameters(): p.requires_grad = False
        for p in model.seg.parameters(): p.requires_grad = True
        for p in model.yolo_layers.parameters(): p.requires_grad = True

    epoch_loss = 0
    
    # Fake Loader Loop
    for i in range(10): # Replace with 'for batch in loader:'
        # Simulate Batch
        batch = ds[0] 
        left = batch['left'].unsqueeze(0).to(DEVICE)
        right = batch['right'].unsqueeze(0).to(DEVICE)
        targets = {k: v.unsqueeze(0).to(DEVICE) if v is not None else None for k,v in batch['targets'].items()}
        
        optimizer.zero_grad()
        
        with autocast():
            preds = model(left, right)
            loss, logs = criterion(preds, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        if i % 5 == 0: print(f"Step {i} | Loss: {loss.item():.4f} | {logs}")

print("🚦 Starting Stage 1: Geometry Training...")
train_epoch(model, None, stage='stereo_only')

print("🚦 Starting Stage 2: Semantic Distillation...")
train_epoch(model, None, stage='distill_heads')

In [ ]:
# === CELL 6: HAILO COMPATIBILITY CHECK ===
def export_onnx():
    print("📦 Exporting to ONNX for Hailo...")
    model.eval()
    
    # Create Dummy Inputs (1 Channel Grayscale)
    dummy_l = torch.randn(1, 1, 480, 640, device=DEVICE)
    dummy_r = torch.randn(1, 1, 480, 640, device=DEVICE)
    
    try:
        torch.onnx.export(
            model,
            (dummy_l, dummy_r),
            "hexapod_fused.onnx",
            input_names=["left", "right"],
            output_names=["disp", "seg", "det_s8", "det_s16", "det_s32"],
            opset_version=11
        )
        print("✅ ONNX Export Successful! Ready for Hailo Dataflow Compiler.")
    except Exception as e:
        print(f"❌ Export Failed: {e}")

export_onnx()